# Fase 4 · SQL e FAISS
Plano Pydantic para SELECT parametrizado. T15 usa um corpus de políticas fictícias e FastEmbed local. T24 demonstra uma falha e uma reformulação com o mesmo índice e top-k, primeiro em memória e ao final sobre o índice salvo em disco (R22/R23).

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import criar_cenario
from pedro.query import planejar, executar
cenario = criar_cenario(Path('storage/notebooks/catalogos'))
pergunta = 'ofertas PEC-01-0001'
plano = planejar(pergunta)
print(plano.model_dump())
print(executar(cenario, pergunta, plano))

{'objetivo': 'ofertas', 'produto_id': 'PEC-01-0001', 'limite': 20}
{'pergunta': 'ofertas PEC-01-0001', 'plano': {'objetivo': 'ofertas', 'produto_id': 'PEC-01-0001', 'limite': 20}, 'sql': 'SELECT o.id,o.part_id,o.supplier_id,s.name AS fornecedor,o.estimated_brl_cents,o.shipping_brl_cents,o.stock_qty,o.lead_days FROM offers o JOIN suppliers s ON s.id=o.supplier_id WHERE o.part_id=? ORDER BY o.supplier_id LIMIT ?', 'parametros': ['PEC-01-0001', 20], 'linhas': [{'id': 'PEC-01-0001-BR-05', 'part_id': 'PEC-01-0001', 'supplier_id': 'BR-05', 'fornecedor': 'Litoral Parts', 'estimated_brl_cents': 37020, 'shipping_brl_cents': 7940, 'stock_qty': 0, 'lead_days': 12}, {'id': 'PEC-01-0001-BR-06', 'part_id': 'PEC-01-0001', 'supplier_id': 'BR-06', 'fornecedor': 'CentroPeças', 'estimated_brl_cents': 143892, 'shipping_brl_cents': 8523, 'stock_qty': 67, 'lead_days': 2}, {'id': 'PEC-01-0001-INT-06', 'part_id': 'PEC-01-0001', 'supplier_id': 'INT-06', 'fornecedor': 'Atlantic Repair Parts', 'estimated_brl_cen

## R21 · text-to-SQL: o modelo escreve a consulta e o motor decide
A célula anterior mostra a versão conservadora (um plano com dois modelos fixos de SQL), que é a que a ferramenta do agente usa. Aqui está a versão com modelo: gera um `SELECT`, a forma dele é validada, e o **autorizador do SQLite** só deixa ler as tabelas e colunas permitidas. **Duplo de teste:** o gerador devolve consultas escritas à mão, a primeira com um erro de propósito, para ver a correção. A execução com o modelo real está em `docs/rastros/fase4_sql_llm.json`.

In [3]:
import json
from langchain_core.runnables import RunnableLambda
from pedro.query import ConsultaRejeitada, consultar_sql, criar_gerador_sql, descrever_esquema, responder_sql
print(descrever_esquema())
roteiro = iter(['SELECT * FROM suppliers WHERE domestic = 1',
                'SELECT id, name FROM suppliers WHERE domestic = 1 ORDER BY id LIMIT 3'])
gerador = criar_gerador_sql(RunnableLambda(lambda p: json.dumps({'sql': next(roteiro), 'explicacao': 'fornecedores nacionais'})))
resultado = responder_sql(cenario, 'Quais fornecedores nacionais existem?', gerador)
for tentativa in resultado['tentativas']:
    print('SQL :', tentativa['sql'])
    print('erro:', tentativa['erro'])
print('estado:', resultado['estado'], '| linhas:', resultado['linhas'])
for sql in ('DROP TABLE offers', 'SELECT 1; DELETE FROM offers', 'SELECT name FROM sqlite_master',
            'SELECT id, website FROM suppliers', 'SELECT count(*) FROM parts a, parts b, parts c'):
    try:
        consultar_sql(cenario, sql)
        print('EXECUTADA:', sql)
    except ConsultaRejeitada as erro:
        print('rejeitada:', sql, '->', erro)

- parts(id, sku, category_id, family, name, description, manufacturer, part_kind, material, color, voltage_v, power_w, weight_g, warranty_months, origin_country, condition)
- categories(id, slug, name)
- suppliers(id, name, country_code, domestic, currency)
- offers(id, part_id, supplier_id, estimated_brl_cents, shipping_brl_cents, stock_qty, availability, lead_days, min_order_qty)
- compatibility(part_id, brand, model_code, year_from, year_to)
- price_history(offer_id, observed_on, estimated_brl_cents)
SQL : SELECT * FROM suppliers WHERE domestic = 1
erro: o motor não autorizou a consulta: leitura de suppliers.website
SQL : SELECT id, name FROM suppliers WHERE domestic = 1 ORDER BY id LIMIT 3
erro: None
estado: ok | linhas: [{'id': 'BR-01', 'name': 'Brava Peças'}, {'id': 'BR-02', 'name': 'Sul Componentes'}, {'id': 'BR-03', 'name': 'Norte Reposição'}]
rejeitada: DROP TABLE offers -> só são admitidas consultas SELECT
rejeitada: SELECT 1; DELETE FROM offers -> só é admitida uma instrução

## T15 · embeddings, FAISS e filtros
As três políticas são sintéticas e citam campos e exemplos da base gerada com a semente 20261107. Cada seção produz vários fragmentos com ID e metadados estáveis; o índice é construído em memória e nenhum pickle é carregado. Os exemplos datados não substituem os dados do cenário ativo.

In [4]:
from collections import Counter
from pedro.policy_demo import fragmentos_demo, criar_indice_demo, impressao_corpus
from pedro.rag import consultar
fragmentos = fragmentos_demo()
indice_demo = criar_indice_demo()
print('corpus_sha256', impressao_corpus())
print('fragmentos_por_tema', dict(Counter(d.metadata['tema'] for d in fragmentos)))
print('dimension', indice_demo.index.d, 'total', indice_demo.index.ntotal)
perguntas = [
    ('Quais fotografias devem ser anexadas para relatar uma carcaça quebrada ao abrir a caixa?', 'devolucoes'),
    ('Como se verifica a compatibilidade elétrica antes de instalar uma peça?', 'garantia'),
    ('O que acontece se o rastreamento não muda depois do despacho?', 'entrega'),
    ('Quantos meses de garantia fictícia tem o compressor PEC-01-0001?', 'garantia'),
    ('Qual oferta de PEC-01-0001 está descontinuada e sem estoque?', 'entrega'),
]
for texto, tema in perguntas:
    geral = consultar(indice_demo, texto, k=1)[0]
    filtrado = consultar(indice_demo, texto, k=1, filtro={'tema': tema},
                        fetch_k=indice_demo.index.ntotal)[0]
    print('pergunta', texto)
    print('sem_filtro', geral['fonte_id'], geral['metadados']['secao'], round(geral['score'], 3))
    print('com_filtro', filtrado['fonte_id'], filtrado['metadados']['secao'], round(filtrado['score'], 3))
    print('fragmento', filtrado['texto'][:180].replace('\n', ' '))
print('filtro_sem_coincidencias', consultar(indice_demo, perguntas[0][0],
      filtro={'tema': 'inexistente'}, fetch_k=indice_demo.index.ntotal))

/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


corpus_sha256 027051d3ca4ed08b2d98130ce4ee5e1f3393975e49da113b1eac7ef81504ff8d
fragmentos_por_tema {'devolucoes': 25, 'garantia': 25, 'entrega': 25}
dimension 384 total 75
pergunta Quais fotografias devem ser anexadas para relatar uma carcaça quebrada ao abrir a caixa?
sem_filtro FIC-DEV-01-S03-C01 3. Dano visível e conteúdo incompleto 9.285
com_filtro FIC-DEV-01-S03-C01 3. Dano visível e conteúdo incompleto 9.285
fragmento O dano visível ao abrir a caixa, como uma carcaça quebrada, um conector dobrado ou um saco de parafusos faltante, deve ser notificado dentro de **7 dias corridos** desde a entrega 
pergunta Como se verifica a compatibilidade elétrica antes de instalar uma peça?
sem_filtro FIC-DEV-01-S02-C01 2. Devolução de uma peça sem instalar 11.902
com_filtro FIC-GAR-01-S06-C01 6. Situações fora desta cobertura fictícia 13.61
fragmento Um curto-circuito causado por conectar uma peça de voltagem distinta, um cabo cortado durante a instalação ou uma carcaça modificada não são class

## T24 · falha observada e reformulação
Sobre o mesmo índice sintético e com top-4, uma pergunta coloquial sobre várias caixas não recupera a regra de entregas parciais. A resposta existe em `entrega.md`, seção 5. A intenção é reformulada sem incluir a resposta. O filtro por tema é um segundo teste, separado da comparação antes/depois. A distância só é interpretada como L2 porque o índice informa essa métrica. Esta evidência local não substitui o índice preexistente exigido por R22.

In [5]:
from pedro.policy_demo import FRAGMENTO_T24_ESPERADO, PERGUNTA_T24_ORIGINAL, PERGUNTA_T24_REESCRITA
from pedro.rag import diagnosticar_reescrita
original, reescrita, esperado = PERGUNTA_T24_ORIGINAL, PERGUNTA_T24_REESCRITA, FRAGMENTO_T24_ESPERADO
print('original :', original)
print('reescrita:', reescrita)
relatorio = diagnosticar_reescrita(indice_demo, original, reescrita, esperado, k=4)
print('métrica', relatorio['metrica'], 'fragmentos', relatorio['total_fragmentos'],
      'top_k', relatorio['k'], 'fetch_k', relatorio['fetch_k'])
print('fonte_esperada', esperado, relatorio['evidencia_esperada']['metadados'])
for etapa in ('antes', 'depois'):
    print(etapa, 'posição', relatorio['posicao_' + etapa])
    for fragmento in relatorio[etapa]:
        print(fragmento['posicao'], fragmento['fonte_id'],
              fragmento['metadados']['fonte'], fragmento['metadados']['secao'],
              round(fragmento['score'], 3))
print('correcao_observada', relatorio['correcao_observada'])
com_filtro = consultar(indice_demo, original, k=4, filtro={'tema': 'entrega'},
                      fetch_k=indice_demo.index.ntotal)
print('filtro_entrega', [(x['fonte_id'], round(x['score'], 3)) for x in com_filtro])
assert relatorio['correcao_observada']
assert any(x['fonte_id'] == esperado for x in com_filtro)

original : Posso receber em várias caixas?
reescrita: Quando a política de entrega admite um envio parcial das peças solicitadas?
métrica L2 fragmentos 75 top_k 4 fetch_k 20
fonte_esperada FIC-ENT-01-S05-C01 {'id': 'FIC-ENT-01-S05-C01', 'origem': 'ficticio_demo', 'tema': 'entrega', 'secao': '5. Entrega parcial e estoque', 'fonte': 'entrega.md', 'versao': '1.1'}
antes posição None
1 FIC-GAR-01-S01-C01 garantia.md 1. Propósito e peças cobertas 13.556
2 FIC-ENT-01-S04-C01 entrega.md 4. Recebimento e verificação imediata 13.598
3 FIC-DEV-01-S03-C01 devolucoes.md 3. Dano visível e conteúdo incompleto 13.891
4 FIC-ENT-01-S05-C00 entrega.md 5. Entrega parcial e estoque 14.344
depois posição 1
1 FIC-ENT-01-S05-C01 entrega.md 5. Entrega parcial e estoque 8.566
2 FIC-ENT-01-S01-C01 entrega.md 1. Alcance dos prazos simulados 10.385
3 FIC-ENT-01-S04-C01 entrega.md 4. Recebimento e verificação imediata 10.765
4 FIC-DEV-01-S05-C01 devolucoes.md 5. Inspeção e resolução 11.55
correcao_observada True
f

## R22 · índice FAISS existente: é carregado, não construído
O índice de `knowledge/politicas/` foi criado uma vez com `scripts/construir_indice_politicas.py` e está salvo com o seu manifesto. Aqui ele só é **carregado** do disco: primeiro conferem-se o manifesto e as impressões digitais, e sem confiança explícita o docstore não é aberto. Depois repete-se T24 sobre o índice carregado. O corpus é fictício e próprio.

In [6]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from pedro.rag import carregar_indice, inspecionar_indice
pasta = Path('knowledge/politicas')
meta = inspecionar_indice(pasta)
print({k: meta[k] for k in ('origem', 'criado_utc', 'embedding_model', 'dimension', 'metrica')})
print('fragmentos no manifesto:', len(meta['documentos']), '| sha256:', meta['sha256'])
print('corpus sem mudanças desde a indexação:', meta['corpus_sha256'] == impressao_corpus())
embeddings = FastEmbedEmbeddings(model_name=meta['embedding_model'], doc_embed_type='passage')
try:
    carregar_indice(pasta, embeddings, origem_confiavel=False)
except PermissionError as erro:
    print('sem confiança explícita ->', erro)
existente, _ = carregar_indice(pasta, embeddings, origem_confiavel=True)
print('carregado do disco:', existente.index.ntotal, 'fragmentos')
relatorio = diagnosticar_reescrita(existente, original, reescrita, esperado, k=4)
for etapa in ('antes', 'depois'):
    print(etapa, 'posição', relatorio['posicao_' + etapa],
          [(f['fonte_id'], round(f['score'], 3)) for f in relatorio[etapa]])
assert relatorio['correcao_observada']

{'origem': 'ficticio_demo: corpus próprio em knowledge/politicas_demo, indexado por scripts/construir_indice_politicas.py', 'criado_utc': '2026-11-11T17:47:35+00:00', 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'dimension': 384, 'metrica': 'L2'}
fragmentos no manifesto: 75 | sha256: {'index.faiss': '955831a2617981b0700d4b339a5af7d182220f64192570eb507aa9ee8c761fc6', 'index.pkl': '8b57a12da117e4d3181a74fb55a4353fe4c4e41247268e0a2164cc22113550ea'}
corpus sem mudanças desde a indexação: True


sem confiança explícita -> é necessária confiança explícita para desserializar index.pkl
carregado do disco: 75 fragmentos
antes posição None [('FIC-GAR-01-S01-C01', 13.556), ('FIC-ENT-01-S04-C01', 13.598), ('FIC-DEV-01-S03-C01', 13.891), ('FIC-ENT-01-S05-C00', 14.344)]
depois posição 1 [('FIC-ENT-01-S05-C01', 8.566), ('FIC-ENT-01-S01-C01', 10.385), ('FIC-ENT-01-S04-C01', 10.765), ('FIC-DEV-01-S05-C01', 11.55)]
